# DBMI AI Workshop

Welcome! In this notebook, you will learn how to talk to AI models using Python. We will use a library called **OpenAI Agents** made by OpenAI.

Even though the Python library come from OpenAI, it can be used for  models from other companies like Google or Anthropic too. For this workshop, we'll stick with OpenAI's models to keep things simple.

Run each cell in order. If you get stuck, just ask an instructor!

## 1. Install the tools

This cell downloads the code we need so Python understands how to build and run AI assistants.

A "Play" button should appear when you hover over the code below.  Press the "Play" button to run the code.

In [ ]:
%pip install -q openai openai-agents

## 2. Get your personal Workshop Key

**Why this exists:** Think of an **'API Key'** like a **password**.

Whenever you send a request to an AI provider like OpenAI, Anthropic, Google, or AWS, you must provide this key. It is what identifies you to their systems and allows them to track your usage and billing.

To keep things simple, we have created a special "Workshop Key" for you to use today. This key acts as your password to access the models without needing to set up your own personal accounts or share credit card info.

**What to do:**
1. Visit [https://dbmi-ai-workshop.dbmi.deno.net/register](https://dbmi-ai-workshop.dbmi.deno.net/register)
2. Enter your email and the code we give you in class. (The code is: DBMI-WORKSHOP-MAY26)
3. **Copy the Workshop Key** that appears on the screen.
4. **Run the code cell below** (click the Play button).
5. **Paste your key** into the text box that appears and press **Enter**.

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = input(
    f"Paste your Workshop Key (from https://dbmi-ai-workshop.dbmi.deno.net/register): "
).strip()

## 3. Connect to the AI service

This cell tells Python to send your requests through our workshop server. The server checks your key and passes your message to OpenAI.

**Keep your key safe:**
1. **Treat it like a password.** If someone else has it, they can use up your balance.
2. **Don't share it** in screenshots or messages.
3. **Safety first:** Our workshop setup makes it easy to reset an API key and limit costs. Each key starts with a small **\$10 limit**. You may be surprised how far that can go if you use the 'nano' and 'mini' models!

In [ ]:
from openai import AsyncOpenAI
from agents import set_default_openai_client, set_tracing_disabled

set_default_openai_client(AsyncOpenAI(
    base_url="https://dbmi-ai-workshop.dbmi.deno.net/v1",
    api_key=os.environ["OPENAI_API_KEY"],
))
set_tracing_disabled(True)

## 4. Build your first AI Agent

An **Agent** is like a tiny digital assistant. You give it a name, a personality (instructions), and choose which model it should use.

AI cost is measured in **tokens** (about 4 characters each). "Input" is what you send the model; "Output" is the reply it thinks of and sends back.

| Model | Cost | Input (\$ per 1M tokens) | Output (\$ per 1M tokens) | Best for... |
|---|---|---|---|---|
| `gpt-5.4-nano` | Cheapest | \$0.20 | \$1.25 | Simple tasks, very fast. |
| `gpt-5.4-mini` | Medium | \$0.75 | \$4.50 | A balanced default; smarter than nano. |
| `gpt-5.5` | Expensive | \$5.00 | \$30.00 | Very complex logic (uses budget fast). |

**Real-world example:** Sending an entire 50,000-word medical chart (\~70,000 input tokens) and getting back a summary (\~1,000 output tokens) would cost roughly:

- **gpt-5.4-nano:** \~1.5¢
- **gpt-5.4-mini:** \~6¢
- **gpt-5.5:** \~38¢

**Pick the smallest model that works for your task!** Your \$10 budget lasts for hundreds of summaries on 'nano', but only about 26 on the biggest model.

In [ ]:
from agents import Agent, Runner

agent = Agent(
    name="Assistant",
    instructions="You are a helpful assistant.",
    model="gpt-5.4-mini",
)

result = await Runner.run(agent, "Write a haiku about a great AI workshop.")
print(result.final_output)

## 5. Give your agent a Tool

AI models are smart, but they are frozen in time—they don't know what's happening in the world *right now*. If you ask about the weather or a current news story, a standard AI has to guess or admit it doesn't know.

You can give an agent a **tool**, which allows it to look up live information. In this example, we'll give one agent a tool to 'check the weather'.

In [ ]:
import urllib.request
from agents import Agent, Runner, function_tool

# 1. Define the Tool with Salt Lake City hardcoded
@function_tool
def get_weather() -> str:
    """Get the current weather for Salt Lake City using the wttr.in service."""
    url = "https://wttr.in/Salt+Lake+City?format=3"
    try:
        with urllib.request.urlopen(url) as response:
            return response.read().decode('utf-8').strip()
    except Exception as e:
        return "Could not find weather for Salt Lake City."

# 2. Setup an agent WITHOUT the tool
basic_agent = Agent(
    name="Basic Assistant",
    instructions="You are a helpful assistant.",
    model="gpt-5.4-mini"
)

# 3. Setup an agent WITH the tool
tool_enabled_agent = Agent(
    name="Tool Assistant",
    instructions="You are a helpful assistant. Use your tools to check live info.",
    model="gpt-5.4-mini",
    tools=[get_weather]
)

print("--- ASKING BASIC AGENT ---")
res1 = await Runner.run(basic_agent, "What is the weather in Salt Lake City right now?")
print(res1.final_output)

print("\n--- ASKING AGENT WITH TOOL ---")
res2 = await Runner.run(tool_enabled_agent, "What is the weather in Salt Lake City right now?")
print(res2.final_output)

## 6. Build an AI Team

For big projects, one AI might get confused if you ask it to do too much at once. Instead, you can build a **team of agents** where each one has a specific job. Think of it like an assembly line:

1. **The Writer** creates a basic story.
2. **The Poet** takes that story and makes it rhyme.
3. **The Translator** takes the poem and changes the language.

By passing the work from one agent to the next, you may get much better results!

In [ ]:
story_writer = Agent(
    name="Story Writer",
    instructions="Write a very short story (3-4 sentences) on the given topic.",
    model="gpt-5.4-mini",
)

poet = Agent(
    name="Poet",
    instructions="Rewrite the given text as a short rhyming poem (4-8 lines).",
    model="gpt-5.4-mini",
)

pig_latin_translator = Agent(
    name="Pig Latin Translator",
    instructions=(
        "Translate the given text into pig latin. "
        "Move the first consonant cluster to the end and add 'ay'. "
        "Words starting with a vowel get 'way' appended. "
        "Preserve line breaks."
    ),
    model="gpt-5.4-mini",
)

# Step 1 — write the story
story = (await Runner.run(story_writer, "a fascinating AI workshop being taken in Salt Lake City")).final_output
print("STORY:\n" + story + "\n")

# Step 2 — turn the story into a poem
poem = (await Runner.run(poet, story)).final_output
print("POEM:\n" + poem + "\n")

# Step 3 — translate the poem into pig latin
pig_latin = (await Runner.run(pig_latin_translator, poem)).final_output
print("PIG LATIN:\n" + pig_latin)

## 7. Challenge: The Medical Summary Team

Now, let's put it all together. We will create a team to handle a medical discharge summary.

1. **The Simplifier:** Takes medical jargon and explains it in plain English.
2. **The Action Plan:** Extracts exactly what the patient needs to do (pills, appointments).
3. **The Translator:** Translates the summary into another language (e.g., Spanish).

In [ ]:
medical_note = """
Patient presents with acute exacerbation of chronic obstructive pulmonary disease (COPD).
Prescribed Prednisone 40mg daily for 5 days and Albuterol HFA inhaler q4h PRN
in addition to existing inhaler regimen.
Follow up with primary care in 1 week. Monitor for new fever or worsening symptoms.
Low threshold to add antibiotics for CAP.
"""

simplifier = Agent(
    name="Simplifier",
    instructions="Translate medical jargon into very simple, plain English.",
    model="gpt-5.4-mini"
)

action_planner = Agent(
    name="Action Planner",
    instructions="Create a bulleted 'To-Do' list for the patient based on the summary.",
    model="gpt-5.4-mini"
)

translator = Agent(
    name="Spanish Translator",
    instructions="Translate the given text into Spanish.",
    model="gpt-5.4-mini"
)

# 1. Simplify
simple_text = (await Runner.run(simplifier, medical_note)).final_output
print("--- SIMPLE SUMMARY ---")
print(simple_text)

# 2. Get the To-Do list
to_do = (await Runner.run(action_planner, simple_text)).final_output
print("\n--- PATIENT TO-DO LIST ---")
print(to_do)

# 3. Translate the list to Spanish
spanish_list = (await Runner.run(translator, to_do)).final_output
print("\n--- LISTA DE TAREAS (SPANISH) ---")
print(spanish_list)